# Running the MCP server in Docker

**Docker packs the server, its Python and its packages into one image, and runs it as a container: An isolated mini-computer on your computer. The client on your machine reaches it over Streamable HTTP, through a port Docker connects to the container.**

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 your computer                   container (image: mcp-server)
 -------------                   -----------------------------
 client.py --&gt; 127.0.0.1:8050 ==&gt; server.py on 0.0.0.0:8050
               (published port)
</pre>

Read `../2-server-stdio-http-sse/client-streamable-http.ipynb` first. The client here is the same one.

## 1. The basics: Image, container and port

- **Image:** A frozen package of a small Linux system, Python, the packages and your code. You build it once, from a `Dockerfile`.
- **Container:** A running copy of an image. It has its own files, processes and network, separate from your computer's.
- **Publishing a port:** A container's ports are closed to your computer until you publish one. `-p 127.0.0.1:8050:8050` connects port 8050 on your computer to port 8050 inside the container. The `127.0.0.1:` part publishes it to your own computer only. A plain `-p 8050:8050` would also publish it to everyone on your network.

## 2. The server, and why it uses 0.0.0.0

<div style="font-size: 0.85em">

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Calculator", host="0.0.0.0", port=8050)


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b


if __name__ == "__main__":
    mcp.run(transport="streamable-http")
```

</div>

It's the Part 2 calculator, on Streamable HTTP, with one change: `host="0.0.0.0"`.

Inside a container, `127.0.0.1` means "this container", not your computer. Docker forwards your requests to the container's own network address, so a server listening only on `127.0.0.1` never receives them. `0.0.0.0` means "listen on every address", so it does.

This was tested for real. The same image, started with the server on `127.0.0.1`, printed `Uvicorn running on http://127.0.0.1:8050`, and the client failed with `httpx.RemoteProtocolError: Server disconnected without sending a response.` The safety that `127.0.0.1` gives a local server comes from the `127.0.0.1:` in `-p` instead.

## 3. The Dockerfile

<div style="font-size: 0.85em">

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY server.py .

EXPOSE 8050

CMD ["python", "server.py"]
```

</div>

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 FROM python:3.12-slim     a small official image with Python 3.12
 WORKDIR /app              work in /app inside the image
 COPY requirements.txt .   copy the package list in first ...
 RUN pip install ...       ... and install it
 COPY server.py .          then the server (the client stays outside)
 EXPOSE 8050               notes the port (doesn't publish it)
 CMD ["python", ...]       what the container runs when it starts
</pre>

The package list is copied and installed before the code, because Docker saves each step as a layer and reuses it. When only `server.py` changes, the rebuild skips the slow `pip install`.

`requirements.txt` pins two packages:

<div style="font-size: 0.85em">

```text
mcp==1.27.0
pydantic-settings==2.14.0
```

</div>

`mcp==1.27.0` matches MCP_env. `pydantic-settings` is pinned because without it, pip installed the newest version, 2.16.1, and the server printed an `IncompleteFieldDefinitionWarning` at startup. Version 2.14.0, the one in MCP_env, prints nothing. Pinning the versions you tested keeps the image from changing under you.

## 4. Building and running it

Docker Desktop must be running. In a terminal in this folder, build the image:

<div style="font-size: 0.85em">

```bash
docker build -t mcp-server .
```

</div>

The key lines of the build output:

<div style="font-size: 0.85em">

```text
#5 [1/5] FROM docker.io/library/python:3.12-slim@sha256:a0f57064cbdcd6285fa45f51237ab0b35ea8ebaed06a1f5973e89e19dfb86704
#6 [2/5] WORKDIR /app
#7 [3/5] COPY requirements.txt .
#8 [4/5] RUN pip install -r requirements.txt
#9 [5/5] COPY server.py .
#10 naming to docker.io/library/mcp-server:latest done
```

</div>

Then start a container from it:

<div style="font-size: 0.85em">

```bash
docker run --rm --name mcp-server -p 127.0.0.1:8050:8050 mcp-server
```

</div>

`--rm` deletes the container when it stops, and `--name` gives it a name you can use in other commands. In a second terminal, `docker ps` lists running containers:

<div style="font-size: 0.85em">

```text
IMAGE        STATUS         PORTS                      NAMES
mcp-server   Up 6 seconds   127.0.0.1:8050->8050/tcp   mcp-server
```

</div>

## 5. Connecting the client

With the container running, the client is exactly Part 2's Streamable HTTP client:

In [1]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

url = "http://127.0.0.1:8050/mcp"

In [2]:
async with streamable_http_client(url) as (read, write, get_session_id):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools_result = await session.list_tools()
        result = await session.call_tool("add", {"a": 2, "b": 3})

print(tools_result.tools[0].name)
print(result.content[0].text)

add
5


## 6. What the container logged

The container's terminal (or `docker logs mcp-server`) printed:

<div style="font-size: 0.85em">

```text
INFO:     Started server process [1]
INFO:     Waiting for application startup.
StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8050 (Press CTRL+C to quit)
Created new transport with session ID: c655f48641284a3fa7f83bc01abfb51e
INFO:     172.17.0.1:45564 - "POST /mcp HTTP/1.1" 200 OK
INFO:     172.17.0.1:45574 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     172.17.0.1:45578 - "GET /mcp HTTP/1.1" 200 OK
INFO:     172.17.0.1:45584 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type ListToolsRequest
INFO:     172.17.0.1:45586 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type CallToolRequest
Terminating session: c655f48641284a3fa7f83bc01abfb51e
INFO:     172.17.0.1:45602 - "DELETE /mcp HTTP/1.1" 200 OK
```

</div>

It's the same request pattern as in Part 2. The requests come from `172.17.0.1`, not `127.0.0.1`. That's Docker's internal network: To the server inside the container, your computer's requests arrive through Docker's network gateway.

## 7. Stopping it

Press `Ctrl+C` in the container's terminal, or run `docker stop mcp-server`. Because of `--rm`, the container is deleted. The image stays, so the next `docker run` starts in a few seconds without rebuilding.

## Say it out loud

> "I containerized the MCP server with a small Dockerfile: Python 3.12 slim, the pinned requirements installed first so the layer is cached, then the server, which runs on Streamable HTTP. Inside the container the server has to listen on 0.0.0.0, because Docker forwards traffic to the container's network address, so 127.0.0.1 there would make it unreachable. I tested that. I publish the port with `-p 127.0.0.1:8050:8050`, so it's only reachable from my own machine, and the client connects to `http://127.0.0.1:8050/mcp` exactly as it would to a local server."